# Fit an analytical model to the simulated magnetic field evolution

In [ ]:
import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.magneto_rotational_physics.magnetic_field_evolution as mre
import utilities.plot_settings

In the following, we fit analytical functions to simulated magnetic-field evolution curves. The functional form we use is the following: 

if $\tau_1 < \tau_2 < \tau_{\rm late}$:

$$B(t) = B_{\rm initial} (1 + t/\tau_1)^{a_1} (1 + t/\tau_2)^{a_2-a_1} (1 + t/\tau_{\rm late})^{a_{\rm late}-a_2}$$ 

if $\tau_1 < \tau_{\rm late} < \tau_2$:

$$B(t) = B_{\rm initial} (1 + t/\tau_1)^{a_1} (1 + t/\tau_{\rm late})^{a_{\rm late}-a_1}$$ 

if $\tau_{\rm late} < \tau_1 < \tau_2$:

$$B(t) = B_{\rm initial} (1 + t/\tau_{\rm late})^{a_{\rm late}}$$ 

where $\tau_1 = A_1  B_{\rm initial}^{b_1}$ and $\tau_2 = A_2 B_{\rm initial}^{b_2}$ and $\tau_{\rm late}$ is a constant.

We consider five different magneto-thermal evolution models which are saved in `mlpoppyns/simulator/magneto_rotational_physics/magneto-thermal_evol_curves`. There you can find README files with more details on the specific magneto-thermal models.
In particular, to select the model shown in this notebook we need to set this in the `mlpoppyns/simulator/config_simulator.py` file under the field `cfg["magneto-thermal_model"]` in line 221. You can choose between: "SLy4_dip-tor_heavy", "BSk24_dip-tor_heavy", "BSk24_dip-tor_light", "BSk24_multi_heavy" and "BSk24_multi_light".

For the magneto-thermal simulations the following set-up was employed: The equation of state is SLy4 with a NS mass of 1.4 Msun and radius of 11.74 km. The impurity parameter in the pasta layer is fixed to 100. For the impurity in the outer and inner crust(excluding the pasta layer), the fits of [Carreau et al.(2020)](https://ui.adsabs.harvard.edu/abs/2020A%26A...640A..77C/abstract) have been used (see Figure 5 in that paper). The envelope model is taken from [Potekhin et al. (2015)](https://ui.adsabs.harvard.edu/abs/2015SSRv..191..239P/abstract). Superfluid and superconducting gap parametrisations are taken from [Ho et al. (2015)](https://ui.adsabs.harvard.edu/abs/2015SciA....1E0578H/abstract): SFB for crustal neutrons, TToa for core neutrons and CCDKp for core protons. 

The initial magnetic field of the simulated curves are: $10^{12}$ G, $10^{13}$ G, $10^{14}$ G, $10^{15}$ G, $3 \times 10^{15}$ G.

In [ ]:
base_path = pathlib.Path("../../")

if cfg["magneto-thermal_model"] == "SLy4_dip-tor_heavy":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e12_Btor1e13.csv")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e13_Btor1e14.csv")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e14_Btor1e15.csv")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "cool_curve_CC_Bdip1e15_Btor1e16.csv")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "cool_curve_CC_Bdip5e15_Btor1e16.csv")
elif cfg["magneto-thermal_model"] == "BSk24_dip-tor_heavy":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e12_H.csv")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e13_H.csv")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e14_H.csv")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e15_H.csv")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_5e15_H.csv")    
elif cfg["magneto-thermal_model"] == "BSk24_dip-tor_light":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e12_L.csv")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e13_L.csv")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e14_L.csv")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_1e15_L.csv")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "50-50_5e15_L.csv")
elif cfg["magneto-thermal_model"] == "BSk24_multi_heavy":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e12_H.csv")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e13_H.csv")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e14_H.csv")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e15_H.csv")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_5e15_H.csv")    
elif cfg["magneto-thermal_model"] == "BSk24_multi_light":
    simB12_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e12_L.csv")
    simB13_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e13_L.csv")
    simB14_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e14_L.csv")
    simB15_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_1e15_L.csv")
    simB5e15_path = base_path.joinpath(cfg["magneto-thermal_path"], "multi_5e15_L.csv")  
else:
    raise ValueError(
            "The magneto-thermal model provided in the configuration file is not meant to be used to compute an analytical fit."
        )

In [ ]:
df_simB12 = pd.read_csv(
    simB12_path,
    delimiter=",",
    header=[0],
)
df_simB12.head()

In [ ]:
df_simB13 = pd.read_csv(
    simB13_path,
    delimiter=",",
    header=[0],
)
df_simB13.head()

In [ ]:
df_simB14 = pd.read_csv(
    simB14_path,
    delimiter=",",
    header=[0],
)
df_simB14.head()

In [ ]:
df_simB15 = pd.read_csv(
    simB15_path,
    delimiter=",",
    header=[0],
)
df_simB15.head()

In [ ]:
df_simB5e15 = pd.read_csv(
    simB5e15_path,
    delimiter=",",
    header=[0],
)
df_simB5e15.head()

In [ ]:
t12_sim = df_simB12["t[yr]"].to_numpy().astype(np.float64)
t13_sim = df_simB13["t[yr]"].to_numpy().astype(np.float64)
t14_sim = df_simB14["t[yr]"].to_numpy().astype(np.float64)
t15_sim = df_simB15["t[yr]"].to_numpy().astype(np.float64)
t5e15_sim = df_simB5e15["t[yr]"].to_numpy().astype(np.float64)

B12_sim = df_simB12["B[G]"].to_numpy().astype(np.float64)
B13_sim = df_simB13["B[G]"].to_numpy().astype(np.float64)
B14_sim = df_simB14["B[G]"].to_numpy().astype(np.float64)
B15_sim = df_simB15["B[G]"].to_numpy().astype(np.float64)
B5e15_sim = df_simB5e15["B[G]"].to_numpy().astype(np.float64)

## Evolution curves from the analytical fit model

In [ ]:
# Define an array with the log10 of the initial magnetic field values for the different cooling curves.
log_B0 = np.array([12, 13, 14, 15, np.log10(5.0e15)])

# Define initial magnetic fields where to evaluate the interpolated cooling curves.
# Note that this is needed now to set the right colors.
log_B0_eval = np.linspace(11.0, 16.0, 6)

# Combine the arrays to find the global min and max values.
combined_values = np.concatenate([log_B0, log_B0_eval])
vmin, vmax = combined_values.min(), combined_values.max()

# Create a colormap and normalize it.
cmap = plt.cm.viridis
norm = Normalize(vmin=vmin, vmax=vmax)

In [ ]:
log_B_initial = combined_values
B_initial = 10**log_B_initial
time = np.logspace(0.0, 9.0, 100)

a1 = cfg["a1"]
a2 = cfg["a2"]
A1 = cfg["A1"]
A2 = cfg["A2"]
b1 = cfg["b1"]
b2 = cfg["b2"]
tau_late = cfg["tau_late"]
    
a_late = cfg["a_late"]

B_asymptotic = 10 ** np.random.normal(
    cfg["B_millisec_mean"], cfg["B_millisec_sigma"], len(B_initial)
)

B_fit = np.zeros((len(B_initial), len(time)))

for i in range(len(B_initial)):
    B_fit[i, :] = mre.magnetic_field_evolution_fit_numpy(
        B_initial[i], time, B_asymptotic[i], a1, a2, A1, A2, b1, b2, tau_late, a_late
    )

In [ ]:
# plot the magnetic field evolution curves
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0, 1.0e9)
ax.set_ylim(1.e8,2.e16)
ax.set_xlabel(r"Time $t$ [yr]")
ax.set_ylabel(r"Magnetic field $B$ [G]")

ax.plot(
    t12_sim,
    B12_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[0])),
    rasterized=True,
)
ax.plot(
    t13_sim,
    B13_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[1])),
    rasterized=True,
)
ax.plot(
    t14_sim,
    B14_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[2])),
    rasterized=True,
)
ax.plot(
    t15_sim,
    B15_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[3])),
    rasterized=True,
)
ax.plot(
    t5e15_sim,
    B5e15_sim,
    linestyle="-",
    linewidth=4,
    color=cmap(norm(log_B0[4])),
    rasterized=True,
)

for i in range(len(B_initial)):
    ax.plot(
        time,
        B_fit[i, :],
        linestyle="--",
        linewidth=4,
        color=cmap(norm(log_B_initial[i])),
        rasterized=True,
        alpha=0.5,
    )

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

plt.grid()

#plt.savefig("B_evol_multi.pdf")

plt.show()